In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv
import time
import chardet
import os


In [2]:

# 실거래 원본
sales_file = "서울시_상권분석_매출_행정동.csv"

# 행정동ID-행정동코드-법정동코드 매핑
edm_mapping_name = "서울시_행정동ID_행정동코드_맵핑.csv"

# 기준날자|행정동코드|평균거래금액|거래건수
apt_name = "서울시_행정동_매출_월단위_202001_202512.csv"


In [4]:

# 법정동 갯수는 467개
# 행정동 갯수는 426개

edm_mapping_df = pd.read_csv(edm_mapping_name, encoding="utf-8-sig")


#print(f"맵핑 갯수:{len(edm_mapping_df)}, 행정동ID 갯수:{len(edm_mapping_df['행정동_ID'].unique())}, 행정동코드 갯수:{len(edm_mapping_df['행정동코드'].unique())}, 법정동코드 갯수: {len(edm_mapping_df['법정동코드'].unique())}")

print(f"맵핑 갯수:{len(edm_mapping_df)}, 행정동ID 갯수:{len(edm_mapping_df['행정동_ID'].unique())}, 행정동코드 갯수:{len(edm_mapping_df['행정동코드'].unique())}")


edm_mapping_df.head(2)


맵핑 갯수:426, 행정동ID 갯수:426, 행정동코드 갯수:426


,행정동_ID,행정동코드,행정동이름
0,11010720,1111051500,청운효자동
1,11010530,1111053000,사직동


In [9]:
# 분기별 행정동 매출을 sum하고 월별로  interpolate 한다.
#
# 1. 기준_년분기_코드, 행정동_코드 기준으로 당월_매출_금액, 당월_매출_건수, 주중_매출_금액,
# 주말_매출_금액, 남성_매출_금액, 여성_매출_금액
# 2. 기준_년분기_코드를 YYYYMM으로 분리
# 3. fillna, interpolate
# 4. YYYYMM, 행정동_코드 기준으로 소팅


apt_price_file = "서울시_아파트_평균가격_행정동.csv"


apt_price_df = pd.read_csv(apt_price_file,
                           #encoding="cp949"
                           encoding="utf-8-sig"
                           ).sort_values(["기준_년분기_코드","행정동_코드"])


cols = [
    "기준_년분기_코드",
    "행정동_코드",
    "행정동_코드_명"
    "서비스_업종_코드",
    "서비스_업종_코드_명"
    "당월_매출_금액",
    "당월_매출_건수",
    "주중_매출_금액",
    "주말_매출_금액",
    "남성_매출_금액",
    "여성_매출_금액"
]




#
# apt_price_df[cols] = (
#     apt_price_df[cols]
#     .fillna(0)
#     .astype(int)
# )
#
# value_cols = [
#     "기준_년분기_코드",
#     "행정동_코드",
#     "행정동_코드_명",
#     "아파트_단지_수",
#     "아파트_면적_66_제곱미터_미만_세대_수",
#     "아파트_면적_66_제곱미터_세대_수",
#     "아파트_면적_99_제곱미터_세대_수",
#     "아파트_면적_132_제곱미터_세대_수",
#     "아파트_면적_165_제곱미터_세대_수",
#     "아파트_평균_면적",
#     "아파트_평균_시가"
# ]
#
#
#
# apt_price_df = apt_price_df[apt_price_df["기준_년분기_코드"] >= 20242]
#
# apt_price_df = apt_price_df[value_cols]
#
# apt_price_df["행정동코드"] = apt_price_df["행정동_코드"] * 100
#
# print(f"갯수: {len(apt_price_df)}")
apt_price_df.head(2)


,기준_년분기_코드,행정동_코드,행정동_코드_명,아파트_단지_수,아파트_면적_66_제곱미터_미만_세대_수,아파트_면적_66_제곱미터_세대_수,아파트_면적_99_제곱미터_세대_수,아파트_면적_132_제곱미터_세대_수,아파트_면적_165_제곱미터_세대_수,아파트_가격_1_억_미만_세대_수,아파트_가격_1_억_세대_수,아파트_가격_2_억_세대_수,아파트_가격_3_억_세대_수,아파트_가격_4_억_세대_수,아파트_가격_5_억_세대_수,아파트_가격_6_억_이상_세대_수,아파트_평균_면적,아파트_평균_시가
5394,20194,11110515,청운효자동,342.0,1667.0,650.0,103.0,107.0,129.0,349.0,950.0,755.0,211.0,122.0,59.0,210.0,70.0,243371110.0
5395,20194,11110530,사직동,98.0,322.0,503.0,443.0,671.0,90.0,42.0,240.0,137.0,158.0,138.0,54.0,1260.0,76.0,313674590.0


In [11]:
# interpolate를 통해서 월단위 데이터 전환

# 분기 -> 월 매핑
quarter_month_map = {
    "1": "03",
    "2": "06",
    "3": "09",
    "4": "12"
}

value_cols = [
    "아파트_단지_수",
    "아파트_면적_66_제곱미터_미만_세대_수",
    "아파트_면적_66_제곱미터_세대_수",
    "아파트_면적_99_제곱미터_세대_수",
    "아파트_면적_132_제곱미터_세대_수",
    "아파트_면적_165_제곱미터_세대_수",
    "아파트_평균_면적",
    "아파트_평균_시가"
]

code_cols = [
    "기준_년분기_코드",
    "행정동_코드_명"
]


# YYYYQ -> YYYYMM 변환
apt_price_df["YYYYMM"] = (
    apt_price_df["기준_년분기_코드"]
    .astype(str)
    .str[:4]
    +
    apt_price_df["기준_년분기_코드"]
    .astype(str)
    .str[-1]
    .map(quarter_month_map)
)

result = []

# 행정동별 월 보간
for code, g in apt_price_df.groupby("행정동_코드"):

    g = g.copy()

    # 날짜 변환
    g["DATE"] = pd.to_datetime(
        g["YYYYMM"],
        format="%Y%m"
    )

    g = g.sort_values("DATE")

    # index 설정
    g = g.set_index("DATE")

    # 월 단위 확장
    monthly = g.resample("MS").asfreq()

    # 수치 컬럼 보간
    monthly[value_cols] = (
        monthly[value_cols]
        .interpolate(method="linear")
    )

    # code 채우기
    monthly[code_cols] = (
        monthly[code_cols]
        .ffill()
    )

    # 행정동코드 유지
    monthly["행정동코드"] = code

    # YYYYMM 생성
    monthly["YYYYMM"] = (
        monthly.index.strftime("%Y%m")
    )

    result.append(monthly)


# 합치기
monthly_apartment_df = (
    pd.concat(result)
    .reset_index(drop=True)
)


monthly_apartment_df[value_cols] = (
    monthly_apartment_df[value_cols]
    .round()
    .fillna(0)
    .astype(int)

)

monthly_apartment_df[code_cols] = (
    monthly_apartment_df[code_cols]
    .ffill()
)

monthly_apartment_df = (
    monthly_apartment_df
    .sort_values(
        ["YYYYMM", "행정동코드"]
    )
    .reset_index(drop=True)
)

monthly_apartment_df[[
    "YYYYMM",
    "행정동코드",
    "행정동_코드_명",
    "아파트_단지_수",
    "아파트_면적_66_제곱미터_미만_세대_수",
    "아파트_면적_66_제곱미터_세대_수",
    "아파트_면적_99_제곱미터_세대_수",
    "아파트_면적_132_제곱미터_세대_수",
    "아파트_면적_165_제곱미터_세대_수",
    "아파트_평균_면적",
    "아파트_평균_시가"]].to_csv(
    apt_name,
    index=False,
    encoding="utf-8-sig")


print(f"갯수: {len(monthly_apartment_df)}")
monthly_apartment_df.head(2)



갯수: 30974


,기준_년분기_코드,행정동_코드,행정동_코드_명,아파트_단지_수,아파트_면적_66_제곱미터_미만_세대_수,아파트_면적_66_제곱미터_세대_수,아파트_면적_99_제곱미터_세대_수,아파트_면적_132_제곱미터_세대_수,아파트_면적_165_제곱미터_세대_수,아파트_가격_1_억_미만_세대_수,아파트_가격_1_억_세대_수,아파트_가격_2_억_세대_수,아파트_가격_3_억_세대_수,아파트_가격_4_억_세대_수,아파트_가격_5_억_세대_수,아파트_가격_6_억_이상_세대_수,아파트_평균_면적,아파트_평균_시가,YYYYMM,행정동코드
0,20194.0,11110515.0,청운효자동,342,1667,650,103,107,129,349.0,950.0,755.0,211.0,122.0,59.0,210.0,70,243371110,201912,11110515
1,20194.0,11110530.0,사직동,98,322,503,443,671,90,42.0,240.0,137.0,158.0,138.0,54.0,1260.0,76,313674590,201912,11110530


In [11]:

from datetime import datetime
from dateutil.relativedelta import relativedelta

start_date = 202001
#end_date = "202512"
middle_date = 202512
#middle_date = (datetime.today() - relativedelta(months=2)).strftime("%Y%m")      # 오늘 기준 이전전달
end_date = (datetime.today() - relativedelta(months=1)).strftime("%Y%m")      # 오늘 기준 이전달

base_file_name = f"../data/서울시_행정동_아파트_월단위_{start_date}_{middle_date}.csv"
file_name = f"../data/서울시_행정동_아파트_월단위_{start_date}_{end_date}.csv"
filled_file_name = f"../data/서울시_행정동_아파트_월단위_월단위_{start_date}_{end_date}_base.csv"



# 2. 데이터 가져오기
base_df = pd.read_csv(base_file_name, encoding="utf-8-sig")

last_df = base_df[base_df["YYYYMM"] == middle_date].copy()

last_df

,YYYYMM,행정동코드,행정동_코드_명,아파트_단지_수,아파트_면적_66_제곱미터_미만_세대_수,아파트_면적_66_제곱미터_세대_수,아파트_면적_99_제곱미터_세대_수,아파트_면적_132_제곱미터_세대_수,아파트_면적_165_제곱미터_세대_수,아파트_평균_면적,아파트_평균_시가
30550,202512,11110515,청운효자동,323,1533,581,83,91,117,69,276146690
30551,202512,11110530,사직동,89,309,158,54,81,14,73,371237211
30552,202512,11110540,삼청동,9,12,19,5,2,14,111,401044841
30553,202512,11110550,부암동,189,712,662,162,46,130,82,259634651
30554,202512,11110560,평창동,294,527,915,314,472,538,124,480839259
...,...,...,...,...,...,...,...,...,...,...,...
30969,202512,11740640,성내1동,363,2527,830,82,15,85,57,252319568
30970,202512,11740650,성내2동,329,2454,429,30,1,12,48,202618082
30971,202512,11740660,성내3동,460,2829,1026,99,40,2,54,212466670
30972,202512,11740685,길동,618,3958,1788,207,75,13,55,227597799


In [12]:

new_rows = []

for month in range(1, 5):  # 1~4
    new_df = last_df.copy()

    new_df["YYYYMM"] = 2026*100 + month

    new_rows.append(new_df)

# 한 번에 추가
base_df = pd.concat([base_df] + new_rows, ignore_index=True)




In [13]:
base_df.to_csv(filled_file_name, index=False, encoding="utf-8-sig")